## 1. Loading the data

Four sheets, one workbook. Before anything else I want to know what I'm actually holding: how many rows, what types, what's missing.

In [1]:
import pandas as pd
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for directory in (current, *current.parents):
        if (directory / "pyproject.toml").exists():
            return directory
    raise FileNotFoundError("Project root containing pyproject.toml not found")

PROJECT_ROOT = find_project_root()

PATH = PROJECT_ROOT / "sample_dataset" / "Synthetic_GNMTC_Vessel_Cost_Risk_Data.xlsx"

xl = pd.ExcelFile(PATH)

print("Sheets in workbook are as:")

for sheet_name in xl.sheet_names:
    print(f"--{sheet_name}")


Sheets in workbook are as:
--README
--Data_Dictionary
--Vessel_Master
--Monthly_Budget_Actual
--Cost_Transactions
--Handover_Deferred_Risk


In [3]:
SHEETS = [
    "Vessel_Master",
    "Monthly_Budget_Actual",
    "Cost_Transactions",
    "Handover_Deferred_Risk",
]

data = {s: pd.read_excel(PATH, sheet_name=s) for s in SHEETS}

for name, df in data.items():
    print(f"Sheet:{name}   rows:{df.shape[0]}  cols:{df.shape[1]}")

    # Nulls are worth seeing per column
    print("\nNulls per column:")
    print(df.isna().sum().to_string())
    print("=="*30)

Sheet:Vessel_Master   rows:14  cols:6

Nulls per column:
Vessel_ID            0
Vessel_Name          0
Vessel_Type          0
Age_Years            0
DWT                  0
Management_Status    0
Sheet:Monthly_Budget_Actual   rows:3472  cols:9

Nulls per column:
Vessel_ID         0
Vessel_Name       0
Period            0
Cost_Category     0
Budget_USD        0
Actual_USD        0
Off_Budget_USD    0
Variance_USD      0
Variance_Pct      0
Sheet:Cost_Transactions   rows:10361  cols:8

Nulls per column:
Transaction_ID       0
Vessel_ID            0
Transaction_Date     0
Cost_Category        0
Description_Theme    0
Amount_USD           0
Urgency              0
Budget_Status        0
Sheet:Handover_Deferred_Risk   rows:10  cols:7

Nulls per column:
Vessel_ID                          0
Vessel_Name                        0
Open_Issues                        0
Overdue_Issues                     0
Estimated_Deferred_Exposure_USD    0
Handover_Risk                      0
Primary_Owner         

In [4]:
vessel_master      = data["Vessel_Master"]          # one row per vessel
monthly_costs      = data["Monthly_Budget_Actual"]  # one row per vessel x month x category
transactions       = data["Cost_Transactions"]      # one row per transaction
deferred_risk      = data["Handover_Deferred_Risk"] # one row per vessel (10 of 14)

## 2. Sanity level checks first

Before comparing any numbers I want to know what the data covers and whether the sheets agree on the basics.

Four things:

1. **What period does this cover, and does every vessel cover the same months?**
- A vessel that only appears for part of the range will look cheap on any annual total.


2. **Do the sheets agree on which vessels exist?**
- Vessel_Master lists 14. If the other sheets disagree, I need to know which ones are missing and from where.

3. **Is the monthly grid complete?**
- `Monthly_Budget_Actual` should hold one row per vessel, per month, per cost category.
- If rows are missing, a gap will read as zero spend when it may only mean "not reported."
 
4. **Do the cost category labels match between the two sheets?**
- If one sheet says "Repairs & Maintenance" and the other has a extra space or different casing, any grouped comparison silently drops rows.

In [11]:
# ---- 1. Period coverage -------------------------------------------------
first_month = monthly_costs["Period"].min()
last_month  = monthly_costs["Period"].max()

# Every month between the first and last, inclusive.
expected_months = (
    (last_month.year - first_month.year) * 12
    + (last_month.month - first_month.month)
    + 1
)
actual_months = monthly_costs["Period"].nunique()


print("Monthly_Budget_Actual period range:")
print(f"  {first_month.date()}  ->  {last_month.date()}")
print(f"  expected distinct months: {expected_months}")
print(f"  actual distinct months:   {actual_months}")
print(f"  gap: {expected_months - actual_months}")

print("=="*20)

print("\nCost_Transactions date range:")
print(f"  {transactions['Transaction_Date'].min()}  ->  {transactions['Transaction_Date'].max()}")


Monthly_Budget_Actual period range:
  2024-01-01  ->  2026-07-01
  expected distinct months: 31
  actual distinct months:   31
  gap: 0

Cost_Transactions date range:
  2024-01-01 00:00:00  ->  2026-07-27 00:00:00


In [13]:
# Does every vessel cover the same months?
months_per_vessel = monthly_costs.groupby("Vessel_ID")["Period"].agg(
    n_months="nunique", first="min", last="max"
)
print("\nMonths covered per vessel:")
print(months_per_vessel.to_string())


Months covered per vessel:
           n_months      first       last
Vessel_ID                                
V1001            31 2024-01-01 2026-07-01
V1002            31 2024-01-01 2026-07-01
V1003            31 2024-01-01 2026-07-01
V1004            31 2024-01-01 2026-07-01
V1005            31 2024-01-01 2026-07-01
V1006            31 2024-01-01 2026-07-01
V1007            31 2024-01-01 2026-07-01
V1008            31 2024-01-01 2026-07-01
V1009            31 2024-01-01 2026-07-01
V1010            31 2024-01-01 2026-07-01
V1011            31 2024-01-01 2026-07-01
V1012            31 2024-01-01 2026-07-01
V1013            31 2024-01-01 2026-07-01
V1014            31 2024-01-01 2026-07-01


In [16]:
# ---- 2. Do the sheets agree on which vessels exist? ---------------------
ids = {
    "vessel_master": set(vessel_master["Vessel_ID"]),
    "monthly_costs": set(monthly_costs["Vessel_ID"]),
    "transactions":  set(transactions["Vessel_ID"]),
    "deferred_risk": set(deferred_risk["Vessel_ID"]),
}

print("\nVessel counts per sheet:")
for name, s in ids.items():
    print(f"  {name:<15} {len(s):>3}")

master = ids["vessel_master"]
print("\nDifferences against Vessel_Master:")
for name, s in ids.items():
    if name == "vessel_master":
        continue
    print(f"  {name:<15} missing: {sorted(master - s)}   extra: {sorted(s - master)}")



Vessel counts per sheet:
  vessel_master    14
  monthly_costs    14
  transactions     14
  deferred_risk    10

Differences against Vessel_Master:
  monthly_costs   missing: []   extra: []
  transactions    missing: []   extra: []
  deferred_risk   missing: ['V1002', 'V1007', 'V1012', 'V1013']   extra: []


In [19]:
# ---- 3. Is the monthly grid complete? -----------------------------------
n_vessels = monthly_costs["Vessel_ID"].nunique()
n_months  = monthly_costs["Period"].nunique()
n_cats    = monthly_costs["Cost_Category"].nunique()

print(f"\nExpected rows in monthly cost"
      f"{n_vessels} x {n_months} x {n_cats} = {n_vessels * n_months * n_cats}")
print(f"Actual rows: {len(monthly_costs)}")

print("=="*20)

# Duplicates on the key would break every later groupby.
key = ["Vessel_ID", "Period", "Cost_Category"]
print(f"Duplicate vessel/month/category rows: {monthly_costs.duplicated(subset=key).sum()}")




Expected rows in monthly cost14 x 31 x 8 = 3472
Actual rows: 3472
Duplicate vessel/month/category rows: 0


In [20]:
# ---- 4. Do the category labels match across sheets? ---------------------
cats_monthly = set(monthly_costs["Cost_Category"].unique())
cats_txn     = set(transactions["Cost_Category"].unique())

print(f"\nCategories in monthly_costs ({len(cats_monthly)}):")
for c in sorted(cats_monthly):
    print(f"  {c!r}")          # repr makes trailing spaces visible

print(f"\nOnly in monthly_costs: {sorted(cats_monthly - cats_txn)}")
print(f"Only in transactions:  {sorted(cats_txn - cats_monthly)}")


Categories in monthly_costs (8):
  'Crew Travel'
  'Crew Wages'
  'Logistics & Freight'
  'Other Operating Cost'
  'Repairs & Maintenance'
  'Safety & Compliance'
  'Spares'
  'Stores'

Only in monthly_costs: []
Only in transactions:  []


## What the above checks showed us:

1. **Period**: 
    - Jan 2024 to Jul 2026 — 31 months, no gaps. 
    - All 14 vessels cover the identical range, so no vessel joined or left mid-period.

2. **Vessel coverage**: 
    - Vessel_Master, Monthly_Budget_Actual and Cost_Transactions all hold the same 14 vessels. 
    - While `Handover_Deferred_Risk` holds only 10 — V1002, V1007, V1012 and V1013 are absent.
    - Go to 1st point of `brain_storming/01_findings.md` for more detailed discussion on this.

3. **Grid completeness**: 
    - 14 × 31 × 8 = 3,472 rows expected, 3,472 present.
    - zero duplicates on vessel/month/category.

4. **Category labels**: 
    - Eight categories, identical sets on both sheets. 
    - No whitespace or casing mismatches, so grouped comparisons between the two are safe.

5. **Nulls**: None, in any of the four sheets.
    - Structural mess like missing rows, blank cells, inconsistent labels, is what I have just checked for, and there is none.

## 3. Do the numbers agree with each other?

- The same spending is recorded twice in this workbook. `Cost_Transactions` and `Monthly_Budget_Actual` both hold information on Actual USD Spent.

- So I add up the transactions myself and compare against the reported figures. The problem statement says not to assume existing totals and formulas to be correct, and this is how I stop assuming.

- Three things I'll be checking here:
    1. Do the transactions add up to the reported actuals?
        - Group transactions by vessel, month and category, sum the amounts, compare against `Actual_USD`.
    2. Is the the variance calculation correct?
        - `Variance_USD` should be `Actual_USD` − `Budget_USD`, and `Variance_Pct` should be that divided by `Budget_USD`.
        - Also I will confirm sign convention for Variance: positive should mean "overspend".
    3. Where does `Off_Budget_USD` sit?

In [25]:
# Give each transaction the month it belongs to, so it can be compared against the monthly summary sheet.

transactions["Period"] = transactions["Transaction_Date"].dt.to_period("M").dt.to_timestamp()

# Add the transactions up to the same grain as the summary sheet.
txn_rollup = (transactions
              .groupby(["Vessel_ID", "Period", "Cost_Category"], as_index=False)["Amount_USD"]
              .sum())

txn_rollup = txn_rollup.rename(columns={"Amount_USD": "Txn_Total"})

print(txn_rollup.head(10))

  Vessel_ID     Period          Cost_Category  Txn_Total
0     V1001 2024-01-01            Crew Travel    3053.14
1     V1001 2024-01-01             Crew Wages   12993.51
2     V1001 2024-01-01    Logistics & Freight    3601.78
3     V1001 2024-01-01   Other Operating Cost    3212.16
4     V1001 2024-01-01  Repairs & Maintenance    7340.54
5     V1001 2024-01-01    Safety & Compliance    3468.31
6     V1001 2024-01-01                 Spares    9119.01
7     V1001 2024-01-01                 Stores    4816.44
8     V1001 2024-02-01            Crew Travel    3828.50
9     V1001 2024-02-01             Crew Wages    8651.63


In [39]:
# Put the two side by side.

# Reason for outer join here is to catch both cases:
# 1. A summary row with no transactions behind it
# 2. A transaction landing in a vessel-month-category the summary doesn't have    
recon = monthly_costs.merge(
    txn_rollup,
    on=["Vessel_ID", "Period", "Cost_Category"],
    how="outer",
    indicator=True,  # It records for each output row, which input table it came from. 3 values: both, left_only, right_only.
)

display(recon.head(8))

recon["Diff"] = recon["Actual_USD"] - recon["Txn_Total"]

# recon["Mismatch"] is a column of True/False
recon["Mismatch"] = recon["Diff"].abs() > 0.01

print("\n\nRow match between the two sheets:")
print(recon["_merge"].value_counts().to_string())
print(f"\nRows compared:  {len(recon)}")
print(f"Mismatched:     {recon['Mismatch'].sum()}")
print(f"Largest gap:    {recon['Diff'].abs().max():,.2f}")
print(f"Total absolute difference: {recon['Diff'].abs().sum():,.2f}")

,Vessel_ID,Vessel_Name,Period,Cost_Category,Budget_USD,Actual_USD,Off_Budget_USD,Variance_USD,Variance_Pct,Txn_Total,_merge
0,V1001,Vessel-A17,2024-01-01,Crew Travel,3353.04,3053.13,0.0,-299.91,-0.089444,3053.14,both
1,V1001,Vessel-A17,2024-01-01,Crew Wages,11938.39,12993.51,0.0,1055.12,0.088380,12993.51,both
2,V1001,Vessel-A17,2024-01-01,Logistics & Freight,3328.60,3601.78,0.0,273.18,0.082071,3601.78,both
3,V1001,Vessel-A17,2024-01-01,Other Operating Cost,3149.37,3212.16,0.0,62.79,0.019937,3212.16,both
4,V1001,Vessel-A17,2024-01-01,Repairs & Maintenance,7459.54,7340.54,0.0,-119.00,-0.015953,7340.54,both
5,V1001,Vessel-A17,2024-01-01,Safety & Compliance,3096.38,3468.31,0.0,371.93,0.120118,3468.31,both
6,V1001,Vessel-A17,2024-01-01,Spares,7015.53,9119.00,0.0,2103.47,0.299831,9119.01,both
7,V1001,Vessel-A17,2024-01-01,Stores,4476.07,4816.44,0.0,340.37,0.076042,4816.44,both




Row match between the two sheets:
_merge
both          3472
left_only        0
right_only       0

Rows compared:  3472
Mismatched:     690
Largest gap:    0.02
Total absolute difference: 10.28


#### Go to 2nd point of `brain_storming/01_findings.md` for more detailed discussion on above result.

In [48]:
off_budget_rows = recon[recon["Off_Budget_USD"] > 0]

print(f"Rows with off-budget spend: {len(off_budget_rows)}")
print(f"Total off-budget as per monthly budget sheet:           {off_budget_rows['Off_Budget_USD'].sum():,.2f}")
print(f"Total off-budget as per transaction sheet:              {transactions[transactions['Budget_Status'] == 'Non-Budgeted']['Amount_USD'].sum():,.2f}")


Rows with off-budget spend: 228
Total off-budget as per monthly budget sheet:           106,513.80
Total off-budget as per transaction sheet:              711,718.50


In [56]:
nb_counts  = transactions[transactions["Budget_Status"] == "Non-Budgeted"]["Description_Theme"].value_counts()
all_counts = transactions["Description_Theme"].value_counts()

theme_split = pd.DataFrame({
    "NonBudg_Txns": nb_counts,
    "Total_Txns":   all_counts,
})
theme_split["NonBudg_Pct_Txns"] = (
    theme_split["NonBudg_Txns"] / theme_split["Total_Txns"] * 100
).round(1)

print(theme_split.sort_values("NonBudg_Pct_Txns", ascending=False).head(10).to_string())

                       NonBudg_Txns  Total_Txns  NonBudg_Pct_Txns
Description_Theme                                                
Survey preparation               19         440               4.3
Generator components             12         301               4.0
Safety stores                    10         304               3.3
Navigation sensor                 8         263               3.0
Local delivery                   12         448               2.7
Regular wages                    17         663               2.6
Sea freight                      11         431               2.6
Crew change travel               16         634               2.5
Emergency crew travel            16         644               2.5
Valve overhaul                    8         338               2.4


### I expected non-budgeted transactions to cluster on unplanned-sounding work, but they don't.
### "Emergency crew travel" for e.g., by name sounds like an unplanned work hence unbudgeted, but it doesn't come in top 8 contributors  

In [62]:
# Non-budgeted transaction count per vessel, including vessels with none.
nb_by_vessel = (
    transactions
    .assign(is_nonbudg=transactions["Budget_Status"] == "Non-Budgeted")
    .groupby("Vessel_ID")["is_nonbudg"]
    .agg(NonBudg_Txns="sum", Total_Txns="count")
)
nb_by_vessel["NonBudg_Pct"] = (
    nb_by_vessel["NonBudg_Txns"] / nb_by_vessel["Total_Txns"] * 100
).round(2)

# print(nb_by_vessel.sort_values("NonBudg_Pct", ascending=False).to_string())

zero_nb = nb_by_vessel[nb_by_vessel["NonBudg_Txns"] == 0].index.tolist()
print(f"\nAs per `Cost_Transactions` sheet, Vessels with zero non-budgeted spend ({len(zero_nb)}):")
print(zero_nb)


As per `Cost_Transactions` sheet, Vessels with zero non-budgeted spend (9):
['V1001', 'V1003', 'V1004', 'V1006', 'V1007', 'V1009', 'V1010', 'V1012', 'V1014']


In [64]:
ob_by_vessel = (
    monthly_costs
    .assign(has_offbudget=monthly_costs["Off_Budget_USD"] > 0)
    .groupby("Vessel_ID")
    .agg(
        OffBudg_Rows=("has_offbudget", "sum"),
        Total_Rows=("has_offbudget", "count"),
        OffBudg_USD=("Off_Budget_USD", "sum"),
    )
)
ob_by_vessel["OffBudg_Pct_Rows"] = (
    ob_by_vessel["OffBudg_Rows"] / ob_by_vessel["Total_Rows"] * 100
).round(2)

# print(ob_by_vessel.sort_values("OffBudg_Rows", ascending=False).to_string())

zero_ob = ob_by_vessel[ob_by_vessel["OffBudg_Rows"] == 0].index.tolist()
print(f"\nAs per `Monthly_Budget_Actual` sheet, Vessels with no off-budget rows ({len(zero_ob)}):")
print(zero_ob)

same = set(zero_nb) == set(zero_ob)
print(f"\nVVIMP: Same vessels on both sheets? {same}")


As per `Monthly_Budget_Actual` sheet, Vessels with no off-budget rows (9):
['V1001', 'V1003', 'V1004', 'V1006', 'V1007', 'V1009', 'V1010', 'V1012', 'V1014']

VVIMP: Same vessels on both sheets? True


In [67]:
# Which vessels carry non-budgeted spend, per the transactions sheet.
nb_vessels = set(transactions.loc[transactions["Budget_Status"] == "Non-Budgeted", "Vessel_ID"])

# Build one row per vessel with everything needed to compare the groupings.
vessel_view = vessel_master[["Vessel_ID", "Management_Status", "Age_Years", "Vessel_Type"]].copy()

vessel_view["Has_NonBudg"]  = vessel_view["Vessel_ID"].isin(nb_vessels)

vessel_view["In_Deferred"]  = vessel_view["Vessel_ID"].isin(deferred_risk["Vessel_ID"])

vessel_view = vessel_view.merge(
    deferred_risk[[
        "Vessel_ID",
        "Estimated_Deferred_Exposure_USD",
        "Handover_Risk",
        "Open_Issues",
        "Overdue_Issues",
    ]],
    on="Vessel_ID",
    how="left",
)

print(
    vessel_view
    .sort_values(["Has_NonBudg", "Vessel_ID"], ascending=[False, True])
    .to_string(index=False)
)

Vessel_ID Management_Status  Age_Years      Vessel_Type  Has_NonBudg  In_Deferred  Estimated_Deferred_Exposure_USD Handover_Risk  Open_Issues  Overdue_Issues
    V1002            Active         18 Container Vessel         True        False                              NaN           NaN          NaN             NaN
    V1005            Active         19 Offshore Support         True         True                         74399.47          High         15.0             4.0
    V1008   Handover Review         11  Chemical Tanker         True         True                        123953.70          High          5.0             2.0
    V1011            Active         18     Bulk Carrier         True         True                        129845.28           Low          4.0             5.0
    V1013            Active         11  Chemical Tanker         True        False                              NaN           NaN          NaN             NaN
    V1001            Active         17     Bulk Carr

In [72]:
print("\nNon-budgeted spend  vs  present in deferred sheet:")
print("--"*20)
print(pd.crosstab(vessel_view["Has_NonBudg"], vessel_view["In_Deferred"]).to_string())


Non-budgeted spend  vs  present in deferred sheet:
----------------------------------------
In_Deferred  False  True 
Has_NonBudg              
False            2      7
True             2      3


### From above table, one can infer these:

- Of the 9 vessels without non-budgeted spend, 7 are present in the deferred sheet and 2 are not.
- Of the 5 with non-budgeted spend, 3 are present and 2 are not in deferred sheet.
- Hence having non-budgeted spend, tells us nothing about vessel being assessed for deferred risk.

In [78]:
# ---- Is Variance_USD = Actual - Budget? -------------------------------
expected_variance = monthly_costs["Actual_USD"] - monthly_costs["Budget_USD"]
variance_diff = (monthly_costs["Variance_USD"] - expected_variance).abs()

print("Variance_USD check")
print(f"  rows failing by more than 0.01: {(variance_diff > 0.01).sum()}")
print(f"  largest difference:             {variance_diff.max():,.4f}")
print("=="*20)


# ---- Is Variance_Pct = Variance_USD / Budget_USD? ---------------------
print("\nBudget_USD equal to zero:", (monthly_costs["Budget_USD"] == 0).sum())

expected_pct = monthly_costs["Variance_USD"] / monthly_costs["Budget_USD"]
pct_diff = (monthly_costs["Variance_Pct"] - expected_pct).abs()

print("\nVariance_Pct check")
print(f"  rows failing by more than 0.0001: {(pct_diff > 0.0001).sum()}")
print(f"  largest difference:               {pct_diff.max():,.6f}")
print("=="*20)

# ---- Sign convention, and the fleet baseline ---------------------------
is_overspend = monthly_costs["Actual_USD"] > monthly_costs["Budget_USD"]
n_overspend = is_overspend.sum()
n_rows = len(monthly_costs)

print(f"\nRows where actual exceeds budget: {n_overspend} of {n_rows}"
      f"  ({n_overspend / n_rows * 100:.1f}%)")
print(f"  of those, Variance_USD positive: {(monthly_costs.loc[is_overspend, 'Variance_USD'] > 0).sum()}")

# Baseline for later: any vessel or category is compared against this rate,
# not against 50%.
print(f"\nFleet baseline — share of vessel/month/category rows over budget: "
      f"{n_overspend / n_rows * 100:.1f}%")

Variance_USD check
  rows failing by more than 0.01: 0
  largest difference:             0.0000

Budget_USD equal to zero: 0

Variance_Pct check
  rows failing by more than 0.0001: 0
  largest difference:               0.000000

Rows where actual exceeds budget: 1964 of 3472  (56.6%)
  of those, Variance_USD positive: 1964

Fleet baseline — share of vessel/month/category rows over budget: 56.6%


In [82]:
# Total budget, actual and variance per vessel across all 31 months.
by_vessel = monthly_costs.groupby("Vessel_ID").agg(
    Budget_USD=("Budget_USD", "sum"),
    Actual_USD=("Actual_USD", "sum"),
    Variance_USD=("Variance_USD", "sum"),
)

display(by_vessel)

,Budget_USD,Actual_USD,Variance_USD
Vessel_ID,,,
V1001,1282879.29,1265302.65,-17576.64
V1002,1961704.41,2077744.26,116039.85
V1003,2449960.32,2464407.82,14447.50
V1004,1760763.71,1770781.10,10017.39
V1005,1466793.59,1595610.64,128817.05
V1006,2101280.09,2085612.26,-15667.83
V1007,1443298.06,1436125.78,-7172.28
V1008,1617680.70,1741429.06,123748.36
V1009,1717444.04,1741895.15,24451.11


In [84]:

# Variance as a share of budget, so vessels of different size are comparable.
by_vessel["Variance_Pct"] = (by_vessel["Variance_USD"] / by_vessel["Budget_USD"] * 100).round(1)

print(by_vessel.sort_values("Variance_Pct", ascending=False).round(0).to_string())

           Budget_USD  Actual_USD  Variance_USD  Variance_Pct
Vessel_ID                                                    
V1005       1466794.0   1595611.0      128817.0           9.0
V1008       1617681.0   1741429.0      123748.0           8.0
V1002       1961704.0   2077744.0      116040.0           6.0
V1011       2045919.0   2132806.0       86887.0           4.0
V1013       1227966.0   1278408.0       50442.0           4.0
V1012       2421828.0   2473953.0       52125.0           2.0
V1009       1717444.0   1741895.0       24451.0           1.0
V1014       1604164.0   1622682.0       18519.0           1.0
V1003       2449960.0   2464408.0       14447.0           1.0
V1004       1760764.0   1770781.0       10017.0           1.0
V1010       1663818.0   1664732.0         914.0           0.0
V1007       1443298.0   1436126.0       -7172.0          -0.0
V1006       2101280.0   2085612.0      -15668.0          -1.0
V1001       1282879.0   1265303.0      -17577.0          -1.0


### The five vessels with the highest overspend — V1005, V1008, V1002, V1011 and V1013 — are exactly the five vessels that have non-budgeted transactions.

### Non-budgeted transactions are included in Actual_USD, and the budget is fixed, so adding unplanned spending to a vessel necessarily pushes it over.

In [87]:
# is variance stable over time, or drifting?

by_month = monthly_costs.groupby("Period").agg(
    Budget_USD=("Budget_USD", "sum"),
    Variance_USD=("Variance_USD", "sum"),
)
by_month["Variance_Pct"] = (
    by_month["Variance_USD"] / by_month["Budget_USD"] * 100
).round(2)

display(by_month.round(0))

,Budget_USD,Variance_USD,Variance_Pct
Period,,,
2024-01-01,833550.0,26906.0,3.0
2024-02-01,821868.0,14302.0,2.0
2024-03-01,809264.0,14870.0,2.0
2024-04-01,785557.0,8615.0,1.0
2024-05-01,779834.0,18528.0,2.0
2024-06-01,777630.0,22293.0,3.0
2024-07-01,766858.0,1549.0,0.0
2024-08-01,778853.0,25810.0,3.0
2024-09-01,795540.0,-1516.0,-0.0


In [ ]:
# 2026 stops in July, so restricting all years to Jan-Jul removes any
# advantage or penalty from where the data happens to end.

monthly_costs["Year"] = monthly_costs["Period"].dt.year
monthly_costs["Calendar_Month"] = monthly_costs["Period"].dt.month

def variance_pct(df):
    """Overspend as a percentage of budget for whatever rows are passed in."""
    return df["Variance_USD"].sum() / df["Budget_USD"].sum() * 100

# Like-for-like: 2026 stops in July, so restrict every year to Jan-Jul.
jan_to_jul = monthly_costs[monthly_costs["Calendar_Month"] <= 7]
print("Jan-Jul only, by year:")
print(jan_to_jul.groupby("Year").apply(variance_pct).round(2).to_string())

# Full years, for comparison. 2026 excluded because it is incomplete.
full_years = monthly_costs[monthly_costs["Year"] < 2026]
print("\nFull years, by year:")
print(full_years.groupby("Year").apply(variance_pct).round(2).to_string())

Jan-Jul only, by year:
Year
2024    1.92
2025    2.79
2026    3.17

Full years, by year:
Year
2024    1.64
2025    2.63


In [94]:
# Overspend by vessel, per year, on the like-for-like Jan-Jul window.
vessel_trend = (
    jan_to_jul
    .groupby(["Vessel_ID", "Year"])
    .apply(variance_pct)
    .unstack()
    .round(1)
)

# Change from first year to last, to make the movement visible.
vessel_trend["Change"] = (vessel_trend[2026] - vessel_trend[2024]).round(1)

display(vessel_trend.sort_values("Change", ascending=False))

Year,2024,2025,2026,Change
Vessel_ID,,,,
V1008,6.6,9.3,11.5,4.9
V1007,-1.8,-0.2,2.9,4.7
V1002,3.5,8.2,7.4,3.9
V1013,2.2,7.3,5.9,3.7
V1005,6.4,7.9,9.7,3.3
V1006,-2.5,-1.0,-0.4,2.1
V1011,2.9,7.2,4.3,1.4
V1009,-0.2,0.0,0.7,0.9
V1012,0.6,3.4,1.5,0.9


## Above table says: 5 vessels deteriorated by more than 3 percentage points between 2024 and 2026 while 4.

In [104]:
# Variance by category and year for the two vessels, on the like-for-like window.
pair = jan_to_jul[jan_to_jul["Vessel_ID"].isin(["V1008", "V1007", "V1002"])]

cat_trend = (
    pair
    .groupby(["Vessel_ID", "Cost_Category", "Year"])
    .apply(variance_pct)
    .unstack(level="Year")
    .round(1)
)
cat_trend["Change"] = (cat_trend[2026] - cat_trend[2024]).round(1)

print(cat_trend.sort_values(["Vessel_ID", "Change"], ascending=[True, False]).to_string())

Year                             2024  2025  2026  Change
Vessel_ID Cost_Category                                  
V1002     Spares                  3.1   8.7  18.3    15.2
          Repairs & Maintenance   5.2  22.9  18.3    13.1
          Safety & Compliance     5.5   5.3  15.8    10.3
          Logistics & Freight     8.0   5.0  14.9     6.9
          Stores                 -2.5   4.5  -1.7     0.8
          Crew Travel            -0.1  10.8  -1.9    -1.8
          Crew Wages              4.9   2.0   0.8    -4.1
          Other Operating Cost    0.8   5.0  -9.6   -10.4
V1007     Crew Travel            -2.6  -4.9  10.1    12.7
          Crew Wages             -8.6   1.4   2.1    10.7
          Other Operating Cost   -2.0  -6.5   4.9     6.9
          Repairs & Maintenance   2.6  -2.9   6.3     3.7
          Spares                  5.0  -1.7   8.0     3.0
          Stores                 -2.8   4.5  -1.9     0.9
          Logistics & Freight    -1.5   2.3  -5.1    -3.6
          Safe

In [105]:
# ---- Fleet-wide: which cost categories are deteriorating? ---------------
# Same like-for-like window (Jan-Jul) so 2026 is comparable to the full years.
category_trend = (
    jan_to_jul
    .groupby(["Cost_Category", "Year"])
    .apply(variance_pct)
    .unstack()
    .round(1)
)
category_trend["Change"] = (category_trend[2026] - category_trend[2024]).round(1)

print("Variance % by cost category, fleet-wide:")
print(category_trend.sort_values("Change", ascending=False).to_string())

Variance % by cost category, fleet-wide:
Year                   2024  2025  2026  Change
Cost_Category                                  
Safety & Compliance     1.5   4.0   9.3     7.8
Repairs & Maintenance   2.5   5.9   7.1     4.6
Spares                  2.6   4.9   3.7     1.1
Logistics & Freight     4.1   3.7   4.5     0.4
Stores                  0.8   0.0   0.7    -0.1
Other Operating Cost    2.5  -0.1   2.1    -0.4
Crew Wages              1.0   0.8   0.2    -0.8
Crew Travel             1.7   2.9   0.8    -0.9


In [107]:
# For each category, count how many of the 14 vessels got worse.
vessel_category = (
    jan_to_jul
    .groupby(["Cost_Category", "Vessel_ID", "Year"])
    .apply(variance_pct)
    .unstack()
)
vessel_category["Deteriorated"] = vessel_category[2026] > vessel_category[2024]

spread = vessel_category.groupby("Cost_Category").agg(
    Vessels=("Deteriorated", "sum"),
    Of_Total=("Deteriorated", "count"),
)
spread["Pct"] = (spread["Vessels"] / spread["Of_Total"] * 100).round(0).astype(int)

print("Vessels deteriorating, by category:")
print(spread.sort_values("Vessels", ascending=False).to_string())

Vessels deteriorating, by category:
                       Vessels  Of_Total  Pct
Cost_Category                                
Safety & Compliance         10        14   71
Repairs & Maintenance        9        14   64
Spares                       9        14   64
Crew Travel                  7        14   50
Crew Wages                   7        14   50
Logistics & Freight          7        14   50
Stores                       7        14   50
Other Operating Cost         5        14   36


### Next, I am looking at the relationship between cost variance and deffered exposure details

In [111]:
# One row per vessel: cost performance, deferred exposure, and status.
vessel_summary = monthly_costs.groupby("Vessel_ID").agg(
    Budget_USD=("Budget_USD", "sum"),
    Variance_USD=("Variance_USD", "sum"),
)
vessel_summary["Variance_Pct"] = (
    vessel_summary["Variance_USD"] / vessel_summary["Budget_USD"] * 100
).round(2)

# Left joins so the four vessels absent from the deferred sheet stay visible.
vessel_summary = (
    vessel_summary
    .merge(vessel_master[["Vessel_ID", "Management_Status"]],
           left_index=True, right_on="Vessel_ID")
    .merge(deferred_risk[["Vessel_ID",
                          "Estimated_Deferred_Exposure_USD",
                          "Handover_Risk"]],
           on="Vessel_ID", how="left")
    .set_index("Vessel_ID")
)

print(
    vessel_summary[[
        "Estimated_Deferred_Exposure_USD",
        "Handover_Risk",
        "Variance_Pct",
        "Management_Status",
    ]]
    .sort_values("Estimated_Deferred_Exposure_USD", ascending=False)
    .to_string()
)

           Estimated_Deferred_Exposure_USD Handover_Risk  Variance_Pct Management_Status
Vessel_ID                                                                               
V1009                            169705.52        Medium          1.42   Handover Review
V1004                            168701.75           Low          0.57   Handover Review
V1014                            167023.32          High          1.15            Active
V1006                            159314.94        Medium         -0.75   Handover Review
V1003                            159118.85        Medium          0.59            Active
V1010                            130521.61        Medium          0.05            Active
V1011                            129845.28           Low          4.25            Active
V1008                            123953.70          High          7.65   Handover Review
V1005                             74399.47          High          8.78            Active
V1001                